In [ ]:
# 1. Import the Google Earth Engine (GEE) Python client library.
import ee

# Authenticate the current session with Google Earth Engine.
# This opens a browser-based OAuth2 flow to grant access.
# Authentication only needs to be performed once per machine.
ee.Authenticate()

In [2]:
# 2. Import the GEE library again (safe to re-import) and
#    initialize the Earth Engine session using the high-volume
#    endpoint, which is optimized for handling large numbers of
#    concurrent tile requests from distributed workflows.
import ee

ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

In [3]:
# 3. Define a small Area of Interest (AOI) as a GEE Polygon geometry.
#    The coordinates describe a bounding box located in California's
#    Sierra Nevada region, specified in WGS84 (EPSG:4326) longitude/latitude.
#    The polygon is closed by repeating the first coordinate at the end.
aoi = ee.Geometry.Polygon(
    [[[-120.5, 38.5],     # Southwest corner (lower-left)
      [-120.5, 38.55],    # Northwest corner (upper-left)
      [-120.4, 38.55],    # Northeast corner (upper-right)
      [-120.4, 38.5],     # Southeast corner (lower-right)
      [-120.5, 38.5]]]    # Close the polygon at the starting vertex
)

In [4]:
# 4. Define the temporal range for the Landsat 8 image collection.
#    A 5-year window is used to capture sufficient temporal depth
#    for meaningful trend detection via linear regression.
start_date = '2015-01-01'  # Start of the observation period
end_date = '2019-12-31'    # End of the observation period

def prep_sr_l8(image):
    """Applies Landsat 8 Collection 2 Level-2 scaling factors to convert
    raw digital numbers into physical surface reflectance values."""

    # Select all surface reflectance bands (matching the 'SR_B' prefix)
    # and apply the Collection 2 Level-2 scaling formula:
    #   Reflectance = (DN × 0.0000275) + (−0.2)
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)

    # Replace the original unscaled bands with the newly scaled bands.
    # The third argument (True) enables overwriting existing band names.
    return (image.addBands(optical_bands, None, True))

# Build the Landsat 8 Image Collection by:
#   1. Loading the USGS Landsat 8 Collection 2 Tier 1 Level-2 product
#   2. Filtering to only images that intersect the AOI
#   3. Filtering to the defined 5-year date range
#   4. Applying the surface reflectance scaling function to each image
#   5. Selecting only the NIR band (SR_B5) for regression analysis
ic = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
      .filterBounds(aoi)
      .filterDate(start_date, end_date)
      .map(prep_sr_l8)
      .select(['SR_B5']))

In [5]:
# 5. Write the custom user-defined function (UDF) that will be
#    applied to each spatial chunk of the dataset by robustraster.

# Import Pandas for DataFrame operations within the UDF
import pandas as pd
# Import NumPy for numerical computations and NaN handling
import numpy as np

def compute_regression(df):
    """
    Computes pixel-wise linear regression (slope m, intercept b) of
    the NIR band (SR_B5) over time. Each unique (X, Y) pixel location
    produces one slope and one intercept value, representing the
    temporal trend in NIR reflectance.

    The output DataFrame must have the same number of rows as the input
    so that robustraster can correctly map results back to the xarray grid.
    """

    # Create a copy of the input DataFrame to avoid modifying the original
    df = df.copy()

    # Build a boolean mask identifying rows where both the SR_B5 value
    # and the timestamp are non-null (valid observations only)
    valid_mask = df['SR_B5'].notna() & df['time'].notna()

    # Subset the DataFrame to only valid (non-NaN) observations
    df_valid = df[valid_mask].copy()

    # Convert the timestamp strings to fractional years for regression.
    # First parse to nanosecond integers, then convert:
    #   nanoseconds → seconds (÷ 10^9) → years (÷ seconds-per-year).
    # Using 365.25 accounts for leap years. This makes the slope 'm'
    # interpretable as "change in NIR reflectance per year."
    df_valid['time_years'] = (
        pd.to_datetime(df_valid['time']).astype('int64')
        // 10**9           # Convert nanoseconds to seconds
        / (3600 * 24 * 365.25)  # Convert seconds to fractional years
    )

    # Group all valid observations by their spatial pixel coordinates
    # (X, Y) so that regression is computed independently per pixel.
    grouped = df_valid.groupby(['X', 'Y'])

    # Compute the mean of the independent variable (time) per pixel group
    df_valid['x_mean'] = grouped['time_years'].transform('mean')
    # Compute the mean of the dependent variable (NIR reflectance) per pixel group
    df_valid['y_mean'] = grouped['SR_B5'].transform('mean')

    # Calculate deviations from the group means for both variables.
    # These are used in the covariance and variance formulas below.
    df_valid['dx'] = df_valid['time_years'] - df_valid['x_mean']  # Time deviation
    df_valid['dy'] = df_valid['SR_B5'] - df_valid['y_mean']       # NIR deviation

    # Compute the cross-product of deviations (numerator of covariance)
    df_valid['dx_dy'] = df_valid['dx'] * df_valid['dy']
    # Compute the squared time deviations (denominator / variance of x)
    df_valid['dx_sq'] = df_valid['dx'] ** 2

    # Sum the cross-products per pixel group to get the covariance numerator:
    #   Cov(x, y) = Σ (xᵢ − x̄)(yᵢ − ȳ)
    cov = grouped['dx_dy'].transform('sum')
    # Sum the squared deviations per pixel group to get the variance:
    #   Var(x) = Σ (xᵢ − x̄)²
    var = grouped['dx_sq'].transform('sum')

    # Calculate the regression slope (m) using the OLS formula:
    #   m = Cov(x, y) / Var(x)
    # Replace zero variance with NaN to avoid division-by-zero errors
    # (occurs when a pixel has only one valid observation in time).
    df_valid['m'] = cov / var.replace(0, np.nan)

    # Calculate the regression intercept (b) using:
    #   b = ȳ − m × x̄
    df_valid['b'] = df_valid['y_mean'] - df_valid['m'] * df_valid['x_mean']

    # Initialize the output columns in the original DataFrame with NaN.
    # This ensures rows with no valid data retain NaN in the output.
    df['m'] = np.nan
    df['b'] = np.nan

    # Map the computed slope and intercept values back to their
    # corresponding positions in the full-sized DataFrame.
    df.loc[valid_mask, 'm'] = df_valid['m']
    df.loc[valid_mask, 'b'] = df_valid['b']

    # Forward-fill any remaining NaN values within each pixel group
    # by propagating the first valid regression result. Since m and b
    # are constant per pixel, 'first' retrieves the single unique value.
    df['m'] = df.groupby(['X', 'Y'])['m'].transform('first')
    df['b'] = df.groupby(['X', 'Y'])['b'].transform('first')

    # Return only the two target output columns (slope and intercept).
    # robustraster uses these columns to construct the output raster.
    return df[['m', 'b']]

In [ ]:
# 6. Execute the full computation pipeline using robustraster.

# Import the main run() entry point from the robustraster package
from robustraster import run

# Log the start of the computation
print("Starting regression computation over time...")

# Define the chunking strategy for Dask-backed xarray processing.
# "time": -1 loads ALL time steps into memory for each spatial tile,
#   which is required because regression needs the full time series.
# "X": 256 and "Y": 256 set the spatial tile size to 256×256 pixels,
#   balancing memory usage with computational efficiency.
chunks = {"time": -1, "X": 256, "Y": 256}

# Launch the robustraster processing pipeline with all configuration:
run(
    # The Earth Engine ImageCollection to process
    dataset=ic,

    # Dataset configuration: spatial/projection parameters
    dataset_config={
        'geometry': aoi,        # The AOI polygon defining the spatial extent
        'crs': 'EPSG:3310',     # California Albers Equal Area projection
        'scale': 30,            # Landsat 8 native spatial resolution (30 m)
    },

    # User function configuration: the custom regression UDF
    user_function_config={
        "user_function": compute_regression,  # The function applied to each chunk
    },

    # Performance tuning: Dask chunk dimensions
    function_tuning_config={
        "chunks": chunks,  # Chunking strategy defined above
    },

    # Export configuration: output format and destination
    export_config={
        "mode": "raster",                   # Export results as a raster file
        "file_format": "GTiff",             # GeoTIFF output format
        "output_folder": "Regression_Output",  # Directory for output files
        "vrt": True,                        # Generate a GDAL Virtual Raster mosaic
        "report": True                      # Generate a Dask performance report
    },
)